# 🏦 Consumer Complaint Classification — NLP Project

**Dataset:** [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/)

---
**Part 1 covered:** Data Loading · Cleaning · EDA · Text Preprocessing (clean_text, lemmatization, noun extraction)

**Part 2 covers (this notebook):**
- Label Encoding of target variable
- Train/Test Split
- Feature Engineering: TF-IDF Vectorization
- Feature Engineering: SVD / LSA Dimensionality Reduction
- Saving processed features for Part 3 (Modelling)

---

## 📦 0. Install & Import Libraries

In [ ]:
import json
import re
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from wordcloud import WordCloud

import spacy
import warnings

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

import pickle

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
tqdm.pandas()

# Load SpaCy model
try:
    nlp_model = spacy.load('en_core_web_sm')
except:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp_model = spacy.load('en_core_web_sm')

print('✅ Libraries loaded successfully')

---
## 📂 1. Load Data from GitHub (JSON)

> **How to get the URL:**
> 1. Upload `complaints_as_json.json` to your GitHub repo
> 2. Open the file on GitHub → click **Raw**
> 3. Copy the URL and paste it below
>
> The URL format is:
> `https://raw.githubusercontent.com/<username>/<repo>/main/complaints_as_json.json`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Option A: Load from GitHub (recommended — no Drive upload needed)
# Replace the URL below with your actual raw GitHub URL
# ─────────────────────────────────────────────────────────────────────────────
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/<your-username>/<your-repo>/main/data/complaints_as_json.json'

# Uncomment and run when you have the real URL:
# response = requests.get(GITHUB_RAW_URL)
# raw_data = response.json()

# ─────────────────────────────────────────────────────────────────────────────
# Option B: Load from local file (if running locally or file is already uploaded)
# ─────────────────────────────────────────────────────────────────────────────
data_path = 'complaints_as_json.json'   # adjust path as needed
with open(data_path, 'r') as f:
    raw_data = json.load(f)

df = pd.json_normalize(raw_data)
print(f'Raw data shape: {df.shape}')
df.head(2)

---
## 🧹 PART 1 — (Integrated) Data Cleaning & EDA
*(This section reproduces Part 1's pipeline so the notebook is fully self-contained.)*

In [ ]:
# ── Clean column names ──────────────────────────────────────────────────────
def clean_column_name(col):
    if col.startswith('_source.'):
        return col.replace('_source.', '')
    elif col.startswith('_'):
        return col[1:]
    return col

df.columns = [clean_column_name(col) for col in df.columns]
print('Columns after cleaning:', df.columns.tolist())

In [ ]:
# ── Remove rows with empty complaint text ───────────────────────────────────
df = df[df['complaint_what_happened'].replace('', np.nan).notnull()]

# ── Remove duplicate complaints ─────────────────────────────────────────────
df = df.drop_duplicates(subset=['complaint_what_happened'])

# ── Remove categories with fewer than 500 complaints ────────────────────────
top_categories = df['product'].value_counts()
top_categories = top_categories[top_categories > 500].index
df = df[df['product'].isin(top_categories)]

print(f'After filtering: {df.shape}')
print('Products remaining:', df['product'].unique())

In [ ]:
# ── Merge similar product categories ────────────────────────────────────────
def merge_products(p):
    if 'Credit card' in p:
        return 'Credit card'
    elif 'account' in p:
        return 'Bank account'
    else:
        return p

df['product_cleaned'] = df['product'].apply(merge_products)
print('Final product categories:')
print(df['product_cleaned'].value_counts())

In [ ]:
# ── EDA: Complaints Over Time ────────────────────────────────────────────────
df['date_received'] = pd.to_datetime(df['date_received'])
monthly_complaints = df.set_index('date_received').resample('M').size()

plt.figure(figsize=(18, 5))
plt.plot(monthly_complaints.index, monthly_complaints, color='steelblue')
plt.title('Complaints Over Time')
plt.xlabel('Date')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA: Complaints by Product ───────────────────────────────────────────────
product_complaints = df['product_cleaned'].value_counts()

plt.figure(figsize=(12, 6))
product_complaints.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Complaints by Product (Cleaned)', fontsize=16)
plt.xlabel('Product')
plt.ylabel('Number of Complaints')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Text Cleaning ────────────────────────────────────────────────────────────
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\w*\d\w*', ' ', text)
    return text.strip()

df['clean_text'] = df['complaint_what_happened'].apply(clean_text)
print('Sample cleaned text:')
print(df['clean_text'].iloc[0][:300])

In [ ]:
# ── Lemmatization ────────────────────────────────────────────────────────────
def lemmatize_texts(texts):
    processed = []
    for doc in tqdm(nlp_model.pipe(texts), total=len(texts), desc='Lemmatizing'):
        tokens = [
            token.lemma_
            for token in doc
            if not token.is_stop and not token.is_punct
        ]
        processed.append(' '.join(tokens))
    return processed

df['lemmatized_text'] = lemmatize_texts(df['clean_text'].tolist())
print('Lemmatization done.')

In [ ]:
# ── Noun Extraction ──────────────────────────────────────────────────────────
def extract_nouns(texts):
    nouns = []
    for doc in tqdm(nlp_model.pipe(texts), total=len(texts), desc='Extracting nouns'):
        nouns.append(' '.join([t.text for t in doc if t.pos_ == 'NOUN']))
    return nouns

df['noun_text'] = extract_nouns(df['lemmatized_text'].tolist())
df['complaint_cleaned'] = df['noun_text'].apply(lambda text: text.replace('-PRON-', ''))

# ── WordCloud ────────────────────────────────────────────────────────────────
all_complaints = ' '.join(df['complaint_cleaned'])
wordcloud = WordCloud(max_words=50, background_color='white', colormap='viridis').generate(all_complaints)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Top 50 Words in Complaints')
plt.axis('off')
plt.show()

In [ ]:
# ── Build df_preprocessed (hand-off from Part 1) ────────────────────────────
df_preprocessed = df[[
    'complaint_what_happened',
    'clean_text',
    'lemmatized_text',
    'noun_text',
    'complaint_cleaned',
    'product_cleaned'
]].copy()

print(f'df_preprocessed shape: {df_preprocessed.shape}')
print(f'Null values:\n{df_preprocessed.isna().sum()}')
df_preprocessed.head(2)

---
## 🔧 PART 2 — Feature Engineering
*(Your work starts here)*

### 2.1 Label Encoding (Target Variable)

In [ ]:
le = LabelEncoder()
df_preprocessed['label'] = le.fit_transform(df_preprocessed['product_cleaned'])

# Print mapping for reference
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Label mapping:')
for category, code in sorted(label_mapping.items(), key=lambda x: x[1]):
    print(f'  {code} → {category}')

### 2.2 Train / Test Split (Stratified)

In [ ]:
X = df_preprocessed['lemmatized_text']
y = df_preprocessed['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set size : {len(X_train):,} samples')
print(f'Test set size     : {len(X_test):,} samples')

# Verify class balance
print('\nClass distribution in training set:')
train_dist = pd.Series(y_train).value_counts().sort_index()
for code, count in train_dist.items():
    print(f'  {le.inverse_transform([code])[0]:30s} : {count:,}')

### 2.3 Feature Engineering — Technique 1: TF-IDF Vectorization

> **Why TF-IDF?** It converts text into numeric form while giving higher weight to terms that are important within a specific complaint but rare across the whole corpus — exactly what we need to distinguish product categories.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'TF-IDF matrix shape — Train : {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape — Test  : {X_test_tfidf.shape}')
print(f'Vocabulary size             : {len(tfidf.vocabulary_):,} terms')

# Show top terms by IDF weight (most informative rare terms)
feature_names = tfidf.get_feature_names_out()
idf_scores = tfidf.idf_
top10_idx = np.argsort(idf_scores)[-10:][::-1]
print('\nTop 10 most discriminative terms (highest IDF):')
for i in top10_idx:
    print(f'  {feature_names[i]:30s} IDF = {idf_scores[i]:.4f}')

### 2.4 Feature Engineering — Technique 2: SVD / LSA (Dimensionality Reduction)

> **Why SVD?** TF-IDF creates a very large, sparse matrix. Truncated SVD (LSA) compresses it into a dense, lower-dimensional space that captures *latent semantic relationships* — complaints about the same topic cluster together even if they use different words.

In [ ]:
svd = TruncatedSVD(n_components=300, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd  = svd.transform(X_test_tfidf)

explained_var = svd.explained_variance_ratio_.sum()

print(f'SVD output shape — Train    : {X_train_svd.shape}')
print(f'SVD output shape — Test     : {X_test_svd.shape}')
print(f'Explained variance (300 components): {explained_var:.2%}')

# Plot cumulative explained variance
cumvar = np.cumsum(svd.explained_variance_ratio_)
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, color='steelblue')
plt.axhline(y=0.80, color='red', linestyle='--', label='80% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by SVD Components')
plt.legend()
plt.tight_layout()
plt.show()

### 2.5 Feature Analysis: Distribution of Topics per Category

In [ ]:
# Visualize how the first two SVD components separate categories
import matplotlib.pyplot as plt

train_df_viz = pd.DataFrame({
    'comp1': X_train_svd[:, 0],
    'comp2': X_train_svd[:, 1],
    'category': le.inverse_transform(y_train)
})

# Sample for readability
sample = train_df_viz.groupby('category').apply(lambda x: x.sample(min(200, len(x)), random_state=42)).reset_index(drop=True)

plt.figure(figsize=(12, 7))
for cat in sample['category'].unique():
    mask = sample['category'] == cat
    plt.scatter(sample.loc[mask, 'comp1'], sample.loc[mask, 'comp2'], label=cat, alpha=0.5, s=15)

plt.title('SVD Component 1 vs Component 2 by Product Category')
plt.xlabel('SVD Component 1')
plt.ylabel('SVD Component 2')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## 💾 3. Save Outputs for Part 3 (Modelling)

In [ ]:
# Save feature matrices
import scipy.sparse as sp

# TF-IDF sparse matrices
sp.save_npz('X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('X_test_tfidf.npz',  X_test_tfidf)

# SVD dense matrices
np.save('X_train_svd.npy', X_train_svd)
np.save('X_test_svd.npy',  X_test_svd)

# Labels
np.save('y_train.npy', y_train.values)
np.save('y_test.npy',  y_test.values)

# Fitted transformers (for inference later)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('svd_transformer.pkl', 'wb') as f:
    pickle.dump(svd, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('✅ All files saved successfully!')
print('Files ready for Part 3:')
files = [
    'X_train_tfidf.npz', 'X_test_tfidf.npz',
    'X_train_svd.npy', 'X_test_svd.npy',
    'y_train.npy', 'y_test.npy',
    'tfidf_vectorizer.pkl', 'svd_transformer.pkl', 'label_encoder.pkl'
]
for f in files:
    print(f'  - {f}')

---
## 📊 4. Summary

| Step | Technique | Input | Output Shape |
|------|-----------|-------|--------------|
| Cleaning | Regex + lowercase | raw text | clean_text |
| Preprocessing | SpaCy lemmatization | clean_text | lemmatized_text |
| Preprocessing | Noun extraction | lemmatized_text | noun_text |
| Label Encoding | LabelEncoder | product_cleaned | label (int) |
| Split | Stratified 80/20 | X, y | X_train, X_test |
| **Feature Eng. 1** | **TF-IDF (unigrams+bigrams)** | **lemmatized_text** | **(n, 10000)** |
| **Feature Eng. 2** | **TruncatedSVD (LSA, 300)** | **TF-IDF matrix** | **(n, 300)** |

**Next step (Part 3):** Train and evaluate classification models using the saved feature matrices.

---
*End of Part 2 — Feature Engineering*